In [0]:
dbutils.widgets.dropdown(name = 'Environment', defaultValue = 'dev', choices = ['dev','qa','prd'], label = 'Environment')
Env = dbutils.widgets.get("Environment")

In [0]:
goldTableName = f"saleslake_{Env}.gold_{Env}.refinedsales"
print(goldTableName)
silverTableName = f"saleslake_{Env}.silver_{Env}.cleanedsales"
print(silverTableName)

In [0]:
spark.sql(f"""MERGE INTO {goldTableName} AS tgt
USING (
    SELECT * FROM {silverTableName}
        WHERE ingest_ts > (
                    SELECT coalesce(MAX(last_updt_ts),to_timestamp('1990-01-01','yyyy-MM-dd')) 
                    FROM {goldTableName}
                    )
) AS src
ON tgt.sale_id = src.sale_id

WHEN MATCHED AND (
    tgt.product <> src.product  OR
    tgt.category <> src.category OR
    tgt.quantity <> src.quantity OR
    tgt.price <> src.price    OR
    tgt.sale_date <> src.sale_date OR
    tgt.region <> src.region
)
THEN UPDATE SET
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    sale_id, product, category, quantity, price, sale_date, region,
    initial_load_ts,
    last_updt_ts
)
VALUES (
    src.sale_id,
    src.product,
    src.category,
    src.quantity,
    src.price,
    src.sale_date,
    src.region,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
%sql
--SELECT * FROM saleslake_dev.gold_dev.refinedsales;